# Matrix-Free Rayleigh Relaxation — Colab quickstart

This notebook is for running the revised Schrödinger ground-state solver on a Google Colab GPU runtime.

Before running it, choose **Runtime → Change runtime type → GPU**.

The notebook uses Colab's preinstalled TensorFlow and installs this repository with `--no-deps`, so Colab's CUDA/TensorFlow stack is not disturbed.

In [ ]:
# Check the GPU assigned by Colab.
!nvidia-smi

## 1. Get the repository

When the new repository exists on GitHub, use the clone cell below.

For now, while testing the starter ZIP, upload `matrix_free_rayleigh_relaxation_starter_repo.zip` using the upload cell instead.

In [ ]:
# OPTION A: use this once the revised repository is public.
# Replace the URL if you choose a different repository name.
REPO_URL = "https://github.com/mahdi-sasar/matrix-free-rayleigh-relaxation.git"

!rm -rf /content/matrix-free-rayleigh-relaxation /content/matrix_free_rayleigh_relaxation
!git clone $REPO_URL /content/matrix-free-rayleigh-relaxation
%cd /content/matrix-free-rayleigh-relaxation

In [ ]:
# OPTION B: use this before the revised repository is public.
# Upload matrix_free_rayleigh_relaxation_starter_repo.zip when prompted.
# Skip this cell if you used OPTION A.

# from google.colab import files
# import zipfile, pathlib, shutil
# uploaded = files.upload()
# zip_name = next(iter(uploaded))
# shutil.rmtree('/content/matrix_free_rayleigh_relaxation', ignore_errors=True)
# with zipfile.ZipFile(zip_name) as zf:
#     zf.extractall('/content')
# %cd /content/matrix_free_rayleigh_relaxation

## 2. Install lightweight dependencies and the local package

Do not reinstall TensorFlow in Colab unless there is a specific reason. The `--no-deps` flag is intentional.

In [ ]:
!python -m pip install -q -r requirements-colab.txt
!python -m pip install -q -e . --no-deps

## 3. Configure TensorFlow and verify precision/device

The examples default to `float64`. Use `--float32` only for exploratory runs or quick debugging.

In [ ]:
import os, json, time
import numpy as np
import pandas as pd
import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

for gpu in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except Exception as exc:
        print("Could not set memory growth:", exc)

tf.keras.backend.set_floatx("float64")
print("Default Keras float:", tf.keras.backend.floatx())

## 4. Small correctness test

This compares the matrix-free stencil against an explicitly assembled dense matrix on tiny grids. It is a sign/stencil sanity check, not a performance benchmark.

In [ ]:
!pytest -q tests/test_small_dense_comparison.py

## 5. First validation run: 3D harmonic oscillator

This is the safest first scientific test because the infinite-domain ground-state energy is known analytically. The finite box and finite grid introduce small shifts, but the result should approach the expected value as the box and grid are refined.

In [ ]:
!python examples/run_harmonic_oscillator_3d.py \
    --n 48 \
    --box 12.0 \
    --alpha 0.25 \
    --sigma 0.9 \
    --tol 1e-8 \
    --max-iter 50000 \
    --out results/colab_harmonic_48

In [ ]:
pd.read_csv("results/colab_harmonic_48/history.csv").tail()

## 6. Hydrogen in a box with an off-grid Coulomb center

This tests the no-smoothing singularity strategy. The Coulomb center is shifted off the grid by half a grid spacing, so the singularity is never sampled directly.

In [ ]:
!python examples/run_hydrogen_box.py \
    --n 64 \
    --box 10.0 \
    --init gaussian \
    --sigma 0.9 \
    --tol 1e-8 \
    --max-iter 100000 \
    --out results/colab_hydrogen_64

In [ ]:
with open("results/colab_hydrogen_64/metadata.json") as f:
    meta = json.load(f)
meta

## 7. Plot a midplane slice

In [ ]:
import matplotlib.pyplot as plt

sl = pd.read_csv("results/colab_hydrogen_64/psi_mid_z.csv", header=None).values
plt.figure(figsize=(5, 4))
plt.imshow(sl.T, origin="lower", extent=[0, 10, 0, 10])
plt.xlabel("X (Bohr)")
plt.ylabel("Y (Bohr)")
plt.colorbar(label="ψ")
plt.title("Hydrogen box, midplane slice")
plt.show()

## 8. Scaling run template

Start modestly. For final paper data, run several grid sizes, repeat runs if needed, and record precision, GPU model, tolerance, iterations, total wall time, and residual.

In [ ]:
!python scripts/run_scaling_hydrogen.py \
    --n-values 64 80 96 112 128 \
    --box 10.0 \
    --tol 1e-8 \
    --sigma 0.9 \
    --max-iter 100000 \
    --out results/colab_hydrogen_scaling.csv

pd.read_csv("results/colab_hydrogen_scaling.csv")

## 9. Optional: save results to Google Drive

Uncomment the following cells for long runs so results survive runtime resets.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/mrsr_results
# !cp -r results/* /content/drive/MyDrive/mrsr_results/

## 10. Larger production run template

For production runs, use `float64` first if the assigned GPU handles it well. If development is too slow, use `--float32` for debugging and reserve `float64` for final tables.

In [ ]:
# Example production-sized run. Uncomment after the smaller runs look correct.
# !python examples/run_hydrogen_box.py \
#     --n 192 \
#     --box 10.0 \
#     --init gaussian \
#     --sigma 0.9 \
#     --tol 1e-8 \
#     --max-iter 200000 \
#     --out /content/drive/MyDrive/mrsr_results/hydrogen_192_float64